In [1]:
# Raw Nairaland
#         |
#         |
# Remove HTML
#         |
#         |
# Remove duplicate threads/comments
#         |
#         |
# Remove spam
#         |
#         |
# Remove phone/email
#         |
#         |
# Normalize whitespace
#         |
#         |
# Handle quotes

In [2]:
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#

In [4]:
import json
import re
import unicodedata

In [5]:
file_path = '/content/drive/MyDrive/thread_data.jsonl'

In [6]:
with open(file_path, 'r') as file:
  for line in file:
      data = json.loads(line)
      break # Only load and print the first line
print(data)

{'category': 'politics', 'title': '2026 Osun Governorship Election Results Update (Photos)', 'link': 'https://www.nairaland.com/8729106/2026-osun-governorship-election-results', 'content': 'Follow this thread for all results from polling units in OSUN as they are been released', 'comments': [{'comment_number': 1, 'comment': 'More results as APC and Accord fight against each other'}, {'comment_number': 2, 'comment': 'See more results as they are pumping'}, {'comment_number': 3, 'comment': 'ADC-9APC-88Accord-51Our Lady School Modakeke'}, {'comment_number': 4, 'comment': 'More results and keep can as I will post every results here'}, {'comment_number': 5, 'comment': 'See even more results from polling units as they are made available to us'}, {'comment_number': 6, 'comment': 'The mergin is very close. APC really want to take over Osun State.'}, {'comment_number': 7, 'comment': 'More results coming and it seems Adeleke in early lead'}, {'comment_number': 8, 'comment': 'Everywhere is silent

In [7]:
# print(f"category{data["category"]}")
# print(f"title: {data["title"]}")
# print(f"link: {data["link"]}")
# print(f"content: {data["content"]}")
# for comment in data["comments"]:
#   print(comment["comment_number"])
#   print(comment["comment"])
#   print()

### General Text cleaning  
  
NB: Converting emails, phone nums and web links into placeholder tags preserves the structural context and sematic meaning of text during text preprocessing

In [8]:
# ---------------------------
# 1. Remove invisible unicode
# ---------------------------

def clean_unicode(text: str) -> str:
    """
    Remove invisible unicode characters while keeping Nigerian characters.
    """

    text = unicodedata.normalize("NFKC", text)

    # Remove zero-width and directional characters
    text = re.sub(r'[\u200b-\u200f\u202a-\u202e]', '', text)

    return text

In [9]:
# ---------------------------
# 2. Normalize whitespace
# ---------------------------

def clean_whitespace(text: str) -> str:
    """
    Remove excessive spaces and broken formatting.
    """

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [10]:
# ---------------------------
# 3. Remove HTML artifacts
# ---------------------------

def remove_html(text: str) -> str:
    """
    Remove basic HTML tags.
    """

    text = re.sub(r'<.*?>', '', text)

    return text

In [11]:
# ---------------------------
# 4. Remove URLs
# ---------------------------

def replace_links(text: str) -> str:
    """
    Replace links instead of deleting them.
    """

    text = re.sub(
        r'https?://\S+|www\.\S+',
        '[LINK]',
        text
    )

    return text

In [12]:
# ---------------------------
# 5. Remove emails
# ---------------------------

def remove_emails(text: str) -> str:

    text = re.sub(
        r'\S+@\S+',
        '[EMAIL]',
        text
    )

    return text

In [13]:
# ---------------------------
# 6. Remove phone numbers
# ---------------------------

def remove_phone_numbers(text: str):

    pattern = r"""
    (?:
        \+?234|
        0
    )
    [\s\-]?
    \d{3}
    [\s\-]?
    \d{3}
    [\s\-]?
    \d{3}
    """

    text = re.sub(
        pattern,
        '[PHONE]',
        text,
        flags=re.VERBOSE
    )

    return text

In [14]:
# ---------------------------
# 7. Remove spam comments
# ---------------------------

SPAM_PATTERNS = [
    "whatsapp",
    "call me",
    "drop your number",
    "check my signature",
    "download now",
    "betting odds",
]


def is_spam(text:str):

    text=text.lower()

    for pattern in SPAM_PATTERNS:
        if pattern in text:
            return True

    return False

In [15]:
# ---------------------------
# 8. Main cleaner
# ---------------------------

def clean_text(text:str):

    if not text:
        return None


    text = clean_unicode(text)

    text = remove_html(text)

    text = replace_links(text)

    text = remove_emails(text)

    text = remove_phone_numbers(text)

    text = clean_whitespace(text)


    # remove useless comments
    if len(text.split()) < 3:
        return None


    if is_spam(text):
        return None


    return text

In [16]:
cleaned_comments = []
for comment in data['comments']:
  cleaned_comment = clean_text(comment['comment'])
  cleaned_comments.append(cleaned_comment)

In [17]:
# cleaned_comments

### Handling quoted comments

In [18]:
# all for handling cases of
# This is bad news for Adeleke..Any governor who loses his capital loses his re-election..These released results are all on Adeleke's stronghold...It's not going well for him..cuz his margins are very little
# bigpicture001:This is bad news for Adeleke..Any governor who loses his capital loses his re-election..These released results are all on Adeleke's stronghold...It's not going well for him..cuz his margins are very littlevery true. Please talk much.


def remove_username(comment: str) -> str:
    """Remove a username appearing at the beginning of a comment."""
    return re.sub(r"^[A-Za-z0-9_]+:", "", comment).strip()


def find_quoted_comment(
    comment: str,
    previous_comments: list[str],
) -> tuple[str, str] | None:
    """
    Check whether `comment` starts with a previous comment.

    Returns:
        (original_comment, reply)

    Example:
        previous = "Hello how are you"
        comment = "Hello how are you I am fine"

        Returns:
            ("Hello how are you", "I am fine")
    """

    for previous in previous_comments:
        previous = remove_username(previous)

        # Avoid matching empty comments
        if not previous:
            continue

        # The current comment must begin with the complete previous comment.
        if comment.startswith(previous) and comment != previous:
            reply = comment[len(previous):].strip()

            if reply:
                return previous, reply

    return None


def handle_quoted_comment(comments: list[str]) -> list[str]:
    """
    Remove usernames and identify comments that quote/repeat
    a previous comment.
    """

    final_cleaned_comments = []
    previous_comments = []

    for comment in comments:
        comment = remove_username(comment)

        quoted = find_quoted_comment(
            comment,
            previous_comments,
        )

        if quoted:
            original, reply = quoted

            final_cleaned_comments.append(
                f'"original/quoted_comment": {original}\n'
                f'"reply": {reply}'
            )
        else:
            final_cleaned_comments.append(comment)

        # Only add the current comment after checking it.
        previous_comments.append(comment)

    return final_cleaned_comments

In [19]:
cleaned_comments = [comment for comment in cleaned_comments if comment is not None]
cleaned_comments = handle_quoted_comment(cleaned_comments)

In [20]:
# cleaned_comments

### Main cleaning Process

In [21]:
# print(f"category{data["category"]}")
# print(f"title: {data["title"]}")
# print(f"link: {data["link"]}")
# print(f"content: {data["content"]}")
# for comment in data["comments"]:
#   print(comment["comment_number"])
#   print(comment["comment"])
#   print()

In [22]:
def process_thread_data(content: str, comments: list[dict[str, str|int]]) -> tuple[str, list[str]]:
  """
  Cleans and processes the content and comments of a discussion thread.

  Args:
    content (str): The main content of the thread.
    comments (list[dict[str, str|int]]): A list of dictionaries, where each dictionary
                                         represents a comment and contains its number and text.

  Returns:
    tuple[str, list[str]]: A tuple containing the cleaned main content (str) and a list
                           of cleaned and processed comments (list of str).
  """
  cleaned_content = clean_text(content)

  cleaned_comments = []
  for comment_item in comments:
    cleaned_single_comment = clean_text(comment_item['comment'])
    if cleaned_single_comment is not None:
      cleaned_comments.append(cleaned_single_comment)

  final_cleaned_comments = handle_quoted_comment(cleaned_comments)

  return cleaned_content, final_cleaned_comments

In [23]:
def format_text(title: str, content: str, comments: list[str]) -> str:
  """Formats the given title, content, and list of comments into a single string with clear headings."""
  return f"Thread Title: {title}\n\nMain Post: {content}\n\nComments:\n\n{"\n\n".join(comments)}"

In [24]:
cleaned_data = []

with open(file_path, "r") as f:
  for line in f:
    # load data
    data = json.loads(line)

    title = data['title']
    category = data['category']
    content = data['content']
    comments = data['comments']

    # clean thread content and comment
    cleaned_content, clean_comment = process_thread_data(content, comments)

    # fomart content and comments
    main_content = format_text(title=title, content=cleaned_content, comments=clean_comment)
    source = "nairaland"
    document_type = "discussion"

    # append result to a list
    cleaned_data.append(
        {"source": source,
         "title": title,
         "category": category,
         "main_content": main_content,
         "document_type": document_type}
    )


### Selecting a fraction of the dataset \
Selected 100 threads from each category for faster embedding

In [31]:
print(f"Number of cleaned threads: {len(cleaned_data)}")

Number of cleaned threads: 3214


In [40]:
categories = {"politics": 0, "education": 0, "sports": 0, "jokes": 0, "romance": 0, "tech": 0}

for thread in cleaned_data:
  category = thread['category']
  if category in list(categories.keys()):
    categories[category] += 1

In [41]:
categories

{'politics': 622,
 'education': 524,
 'sports': 518,
 'jokes': 435,
 'romance': 568,
 'tech': 547}

In [55]:
subset = []
categories = {"politics": 0, "education": 0, "sports": 0, "jokes": 0, "romance": 0, "tech": 0}
for thread in cleaned_data:
  category = thread['category']
  if category in list(categories.keys()):
    if categories[category] < 100:
      subset.append(thread)
      categories[category] += 1

In [56]:
len(subset)

600

### Saving the cleaned data

In [57]:
from google.colab import files

In [58]:
output_filename = "cleaned_threads.json"

In [59]:
with open(output_filename, "w") as f:
  json.dump(subset, f, indent=4)

In [61]:
files.download(output_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>